# 🚀 Full Capstone: Object Detection + Tesla Data Analysis

In [ ]:
# Install libraries
!pip install ultralytics pandas matplotlib seaborn -q

In [ ]:
# Imports
import os, zipfile, shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from google.colab import files

## 🚗 Part 1: Object Detection

In [ ]:
# Create folders
folders = [
    "dataset/images/train",
    "dataset/images/val",
    "dataset/labels/train",
    "dataset/labels/val"
]
for f in folders:
    os.makedirs(f, exist_ok=True)
print("Folders created")

In [ ]:
# Upload dataset zip
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall("raw_data")

print("Dataset extracted")

In [ ]:
# Split data
def move_files(src_img, src_lbl):
    images = os.listdir(src_img)
    split = int(0.8 * len(images))
    for i, img in enumerate(images):
        img_path = os.path.join(src_img, img)
        lbl_path = os.path.join(src_lbl, img.replace(".jpg",".txt").replace(".png",".txt"))
        if i < split:
            shutil.copy(img_path, "dataset/images/train")
            if os.path.exists(lbl_path):
                shutil.copy(lbl_path, "dataset/labels/train")
        else:
            shutil.copy(img_path, "dataset/images/val")
            if os.path.exists(lbl_path):
                shutil.copy(lbl_path, "dataset/labels/val")

move_files("raw_data/images", "raw_data/labels")
print("Data split done")

In [ ]:
# Create YAML
yaml_content = '''
path: dataset
train: images/train
val: images/val

names:
  0: car
  1: bus
  2: truck
'''
with open("dataset/data.yaml", "w") as f:
    f.write(yaml_content)

In [ ]:
# Train model
model = YOLO("yolov8n.pt")
model.train(data="dataset/data.yaml", epochs=20, imgsz=640, batch=16)

In [ ]:
# Evaluate + Predict
model.val()
model.predict(source="dataset/images/val", save=True)

In [ ]:
# Download model
files.download("runs/detect/train/weights/best.pt")

## 📊 Part 2: Tesla Data Analysis

In [ ]:
# Upload CSV
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

In [ ]:
# Cleaning
df = df.drop_duplicates()
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df.fillna(0, inplace=True)

In [ ]:
# EDA Plots
plt.figure()
df.groupby('Year').size().plot(kind='bar')
plt.title("Events per Year")
plt.show()

plt.figure()
df['Deaths'].hist(bins=20)
plt.title("Deaths per Accident")
plt.show()

if 'Model' in df.columns:
    plt.figure()
    df['Model'].value_counts().plot(kind='bar')
    plt.title("Accidents by Model")
    plt.show()

In [ ]:
# Summary
print("Total Accidents:", len(df))
print("Total Deaths:", df['Deaths'].sum())
print("Average Deaths:", df['Deaths'].mean())